# 1. Run all fixed equal-fusion seed-sensitivity jobs

This driver executes the **exact code** from `ADNI_Fixed_Equal_Fusion_Seed_Sensitivity_Configurable.ipynb` for all 15 planned combinations: seeds **17, 42, 73** × outer folds **0-4**.

Safety and restart behaviour:

- each seed/fold runs in a fresh Python process, preventing GPU memory and random-state leakage;
- a valid `_SUCCESS.json` causes that run to be skipped;
- an incomplete run uses the original notebook's `last_epoch_checkpoint.pt` and restores its full RNG state;
- each child process writes a persistent log to the exact fixed-fusion experiment tree on Google Drive;
- a malformed completion marker is treated as an error, never silently overwritten;
- the driver stops on the first failed run by default and records the failure. Re-running **Run all** skips completed combinations and resumes the interrupted one.

Use an **A100 GPU** if available. Choose a high-RAM host only if ordinary system RAM was previously close to exhaustion; GPU memory is the main requirement.


In [ ]:
# ============================================================
# 1. Mount Drive and define the complete run matrix
# ============================================================

from pathlib import Path
from google.colab import drive
import json
import os
import sys


DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive", force_remount=False)

if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive did not mount at /content/drive/MyDrive.")


# These are the complete controlled experiment settings.
RUN_SEEDS = (17, 42, 73)
RUN_FOLDS = (0, 1, 2, 3, 4)

# Safe default: do not cascade through later runs after a genuine failure.
STOP_AFTER_FIRST_FAILURE = True

EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_md050_seed_sensitivity"
TASK_NAME = "mci_prognosis"
EXPERIMENT_ROOT = (
    DRIVE_ROOT
    / "adni_mri"
    / "models"
    / "3mt_tmc_evidential"
    / "experiments"
    / EXPERIMENT_NAME
)
LOG_ROOT = EXPERIMENT_ROOT / "autorun_logs"
STATUS_PATH = EXPERIMENT_ROOT / "autorun_status.json"

LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("Planned seeds:", RUN_SEEDS)
print("Planned folds:", RUN_FOLDS)
print("Total combinations:", len(RUN_SEEDS) * len(RUN_FOLDS))
print("Experiment root:", EXPERIMENT_ROOT)
print("Logs:", LOG_ROOT)


In [ ]:
# ============================================================
# 2. Load the immutable training-program payload and helpers
# ============================================================

import base64
from datetime import datetime
import hashlib
import subprocess
import tempfile
import time
import zlib


CHILD_SOURCE_B64 = "..."
EXPECTED_CHILD_SOURCE_SHA256 = "38db2b714f5645d81ae25e32a06948ef1ddc6fcef7bb3cb39d86762149bf54fb"

CHILD_SOURCE = zlib.decompress(
    base64.b64decode(CHILD_SOURCE_B64)
).decode("utf-8")

actual_child_sha256 = hashlib.sha256(
    CHILD_SOURCE.encode("utf-8")
).hexdigest()
if actual_child_sha256 != EXPECTED_CHILD_SOURCE_SHA256:
    raise RuntimeError("Embedded training-program payload failed its SHA-256 check.")


def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def atomic_write_json(value, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        f".{destination.name}.tmp.{os.getpid()}"
    )
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)


def run_directory(seed, fold):
    return (
        EXPERIMENT_ROOT
        / TASK_NAME
        / f"seed_{seed}"
        / f"fold_{fold}"
    )


def validate_completion_marker(seed, fold):
    directory = run_directory(seed, fold)
    marker_path = directory / "_SUCCESS.json"
    if not marker_path.exists():
        return False

    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
    except Exception as error:
        raise RuntimeError(
            f"Malformed completion marker: {marker_path}"
        ) from error

    expected = {
        "status": "complete",
        "experiment_name": EXPERIMENT_NAME,
        "task": TASK_NAME,
        "fold": int(fold),
        "random_seed": int(seed),
    }
    mismatches = {
        key: {"expected": value, "observed": marker.get(key)}
        for key, value in expected.items()
        if marker.get(key) != value
    }
    if mismatches:
        raise RuntimeError(
            f"Completion marker identity mismatch at {marker_path}: "
            f"{mismatches}"
        )

    required_files = [
        directory / "checkpoints" / "best_validation_auc_checkpoint.pt",
        directory / "history" / "training_history.csv",
        directory / "history" / "training_configuration.json",
        directory / "predictions" / "test_predictions.csv",
        directory / "predictions" / "test_metrics.json",
        directory / "predictions" / "test_metrics_summary.csv",
    ]
    missing = [str(path) for path in required_files if not path.is_file()]
    if missing:
        raise RuntimeError(
            "A completion marker exists but required artifacts are missing: "
            + repr(missing)
        )
    return True


def source_for_run(seed, fold):
    if CHILD_SOURCE.count("SELECTED_FOLD = 0") != 1:
        raise RuntimeError("Training payload fold assignment is ambiguous.")
    if CHILD_SOURCE.count("SELECTED_SEED = 17") != 1:
        raise RuntimeError("Training payload seed assignment is ambiguous.")
    source = CHILD_SOURCE.replace(
        "SELECTED_FOLD = 0",
        f"SELECTED_FOLD = {int(fold)}",
        1,
    )
    source = source.replace(
        "SELECTED_SEED = 17",
        f"SELECTED_SEED = {int(seed)}",
        1,
    )
    return source


def log_tail(path, line_count=80):
    try:
        lines = Path(path).read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
    except FileNotFoundError:
        return "<log file was not created>"
    return "\n".join(lines[-line_count:])


print("Embedded training program verified:", actual_child_sha256)


In [ ]:
# ============================================================
# 3. Run all missing seed/fold combinations unattended
# ============================================================

planned = [
    {"seed": int(seed), "fold": int(fold)}
    for seed in RUN_SEEDS
    for fold in RUN_FOLDS
]

status = {
    "schema_version": 1,
    "experiment_name": EXPERIMENT_NAME,
    "task": TASK_NAME,
    "planned": planned,
    "stop_after_first_failure": bool(STOP_AFTER_FIRST_FAILURE),
    "driver_started_at": now_iso(),
    "driver_finished_at": None,
    "state": "running",
    "current": None,
    "completed_this_invocation": [],
    "skipped_as_complete": [],
    "failures": [],
}
atomic_write_json(status, STATUS_PATH)

print("=" * 72)
print("UNATTENDED FIXED-EQUAL-FUSION SEED/FOLD AUTORUN")
print("=" * 72)

with tempfile.TemporaryDirectory(
    prefix="adni_fixed_equal_fusion_autorun_"
) as temporary_directory:
    temporary_root = Path(temporary_directory)

    for index, item in enumerate(planned, start=1):
        seed = item["seed"]
        fold = item["fold"]
        label = f"seed={seed}, fold={fold}"

        if validate_completion_marker(seed, fold):
            print(f"[{index:02d}/{len(planned):02d}] SKIP complete: {label}")
            status["skipped_as_complete"].append(item)
            atomic_write_json(status, STATUS_PATH)
            continue

        print(f"[{index:02d}/{len(planned):02d}] START: {label}")
        started_at = now_iso()
        status["current"] = {
            **item,
            "started_at": started_at,
        }
        atomic_write_json(status, STATUS_PATH)

        script_path = temporary_root / f"seed_{seed}_fold_{fold}.py"
        script_path.write_text(
            source_for_run(seed, fold),
            encoding="utf-8",
        )
        log_path = LOG_ROOT / f"seed_{seed}_fold_{fold}.log"

        environment = os.environ.copy()
        environment["PYTHONUNBUFFERED"] = "1"
        environment["MPLBACKEND"] = "Agg"

        return_code = None
        process = None
        try:
            with open(log_path, "a", encoding="utf-8", buffering=1) as log:
                log.write("\n" + "=" * 72 + "\n")
                log.write(
                    f"AUTORUN CHILD START {started_at}; {label}\n"
                )
                log.write("=" * 72 + "\n")
                log.flush()

                process = subprocess.Popen(
                    [sys.executable, "-u", str(script_path)],
                    stdout=log,
                    stderr=subprocess.STDOUT,
                    env=environment,
                )
                heartbeat_started = time.monotonic()
                last_heartbeat = heartbeat_started
                while process.poll() is None:
                    time.sleep(30)
                    elapsed = time.monotonic() - heartbeat_started
                    if time.monotonic() - last_heartbeat >= 300:
                        print(
                            f"    still running {label}; "
                            f"elapsed={elapsed / 60:.1f} min; "
                            f"log={log_path}"
                        )
                        last_heartbeat = time.monotonic()
                return_code = int(process.returncode)
                log.write(
                    f"AUTORUN CHILD END {now_iso()}; "
                    f"return_code={return_code}\n"
                )
        except KeyboardInterrupt:
            if process is not None and process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            status["state"] = "interrupted"
            status["current"] = {
                **item,
                "started_at": started_at,
                "interrupted_at": now_iso(),
                "log_path": str(log_path),
            }
            atomic_write_json(status, STATUS_PATH)
            raise

        failure_message = None
        if return_code != 0:
            failure_message = f"Child process returned code {return_code}."
        else:
            try:
                completed = validate_completion_marker(seed, fold)
            except Exception as error:
                completed = False
                failure_message = str(error)
            if not completed and failure_message is None:
                failure_message = (
                    "Child returned successfully but did not create a valid "
                    "_SUCCESS.json marker."
                )

        if failure_message is not None:
            failure = {
                **item,
                "started_at": started_at,
                "failed_at": now_iso(),
                "return_code": return_code,
                "message": failure_message,
                "log_path": str(log_path),
            }
            status["failures"].append(failure)
            status["current"] = None
            status["state"] = "failed"
            atomic_write_json(status, STATUS_PATH)

            print(f"[{index:02d}/{len(planned):02d}] FAILED: {label}")
            print(f"Log: {log_path}")
            print("\nLast log lines:\n")
            print(log_tail(log_path))

            if STOP_AFTER_FIRST_FAILURE:
                raise RuntimeError(
                    f"Autorun stopped safely at {label}. Re-run this "
                    "notebook after addressing the logged error; completed "
                    "runs will be skipped and this run will resume."
                )
            continue

        finished_item = {
            **item,
            "started_at": started_at,
            "completed_at": now_iso(),
            "log_path": str(log_path),
        }
        status["completed_this_invocation"].append(finished_item)
        status["current"] = None
        status["state"] = "running"
        atomic_write_json(status, STATUS_PATH)
        print(f"[{index:02d}/{len(planned):02d}] COMPLETE: {label}")


all_complete = all(
    validate_completion_marker(item["seed"], item["fold"])
    for item in planned
)
status["driver_finished_at"] = now_iso()
status["current"] = None
status["state"] = "complete" if all_complete else "incomplete"
atomic_write_json(status, STATUS_PATH)

print("=" * 72)
if all_complete:
    print("ALL 15 FIXED-EQUAL-FUSION RUNS ARE COMPLETE")
    print("You can now run the controlled learned-vs-fixed consolidation notebook.")
else:
    print("AUTORUN FINISHED WITH ONE OR MORE INCOMPLETE RUNS")
    print("Inspect autorun_status.json and the per-run logs before consolidation.")
print("Status:", STATUS_PATH)
print("=" * 72)
